# Draft Agent — Stage 2 of the split Planner

Researches the brainstorm's gaps and drafts the `PlannerOutput` for one subject. `%run`s
`common` + `research_workflow` (for the `research` tool). Carry-over + anti-fabrication are
enforced by the two code guards. Demo feeds a canned brainstorm so draft stays standalone.

In [ ]:
import os
# Run from backend/ so `%run` and .env resolve regardless of the kernel's launch dir.
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
_DRAFT_WAS_TOP = globals().get("RUN_DEMO", True)  # unique name: avoid clobber when %run by pipeline
RUN_DEMO = False                                   # suppress dependency demos during %run
%run common.ipynb
%run research_workflow.ipynb
RUN_DEMO = _DRAFT_WAS_TOP                          # restore this notebook's own demo flag


In [ ]:
# Cell 4 — Draft agent (stage 2): researches the gaps, drafts PlannerOutput
#
# ModelSettings(tool_choice="required") forces a real tool call on turn 1 —
# the agent cannot produce final output before researching. The SDK resets
# tool_choice to auto after the first tool call (reset_tool_choice=True
# default), so there is no infinite-loop risk.
# Carry-over + anti-fabrication are ALSO enforced in code by Cell 5 guards.

MAX_RESEARCH_CALLS_PER_PLANNER = 3
DRAFT_MAX_TURNS = MAX_RESEARCH_CALLS_PER_PLANNER * 2 + 4  # research + reasoning headroom

DRAFT_PROMPT = f"""\

Persona:
You are a socially aware conversation coach and friendly prep buddy. You help
the user sound curious, relaxed, and easy to talk to.

Task:
A brainstorm stage has already produced first-principles conversation angles and
a list of research gaps for one subject.
Your job: research the gaps. Then carry the INTENT of the best brainstorm angles, not necessarily the exact wording.
You may rewrite, shorten, soften, merge, or drop brainstorm angles to make them
natural, easy to answer, and relationship-appropriate.
Preserve source_url = null for rewritten brainstorm angles.

For reference, today's date is {TODAY}.

You will receive an input message in this format:
Subject:        <subject name>
Phrase:         <the user's original phrase introducing this subject>
Relationship:   <e.g. friend / coworker / first-date / boss>
Research budget: you can and should call the `research` tool up to N times.
Brainstorm (fixed input): subject facet angles, person angles, research gaps.

Carry-over contract (hard rule):
The Brainstorm is fixed input, but its exact wording is not fixed. Carry the
intent of its best angles into your output, with source_url = null. Research ADDS
points — including new angles the findings inspire. You may rewrite, shorten,
soften, merge, or drop brainstorm angles when doing so makes the final point more
natural, easy to answer, and relationship-appropriate. Carry the person angles
especially — they are the hardest to recreate.

Research-inspired synthesis:
Research findings are inspiration, not just facts to relay. When a finding
suggests a better angle, WRITE that new angle — especially hybrids that
connect the news to the person. 
New angles must be personally answerable and must carry source_url when
they lean on a finding.

Freshness rule:
Any point that mentions recent news, current events, or anything
time-sensitive must carry a source_url copied from research output.
Points without a source_url must stand on timeless knowledge and avoid
words like "recently", "current", "right now", "this year", "latest".

Anti-fabrication hard rule:
source_url may ONLY contain a URL copied VERBATIM from a Sources block
returned by the research tool in this conversation. Never write a URL from
memory, and never invent one. If you did not research a fact, do not cite it.

---

METHOD

Step 1 — Read the brainstorm. Decide for each angle intent whether to carry, rewrite,
merge, or drop. 

Step 2 — Research the gaps. You may merge, split, or sharpen the gap
questions to fit the budget. For each, call the research tool ONCE:
research(
query   = "<short Brave search string>",
context = "<why the user cares — about THIS subject and the relationship only; not the person's other interests>",
reason  = "<why this search matters for this conversation>",
)
It is mandatory to call research at least once. Use the full budget when
the gaps deserve it; a focused follow-up on the most promising finding is
often the best use of a spare call.

Step 3 — Draft. Combine three kinds of points:

* brainstorm-derived angles: rewritten or preserved, source_url = null
* researched points: source_url copied VERBATIM from the Sources block
* research-inspired hybrids: new angles the findings sparked, with
  source_url when they lean on the finding
  Rewrite researched news items into personally-answerable angles — the
  other person must be able to respond from experience, preference, or
  opinion.

---

OUTPUT — fill every field of PlannerOutput:

subject
The subject name exactly as given.

framing
2-3 sentences mixing background knowledge and, when useful, researched recent
context. This is the subject's intro paragraph on the final card.

themes
2-5 named clusters of TalkingPoints. Names emerge from the points
themselves. Brainstorm-derived and researched points can mix inside a theme.

themes[].points — each TalkingPoint:
angle      A compact conversation angle — a prep-note bullet, not a
script line to say out loud.
context    Optional short background, 1-2 sentences. For researched
points, include only the necessary details so the user understands
where the angle comes from.
source_url Per the anti-fabrication rule above.

reasoning_notes
One paragraph: Expalins why your output angle are natural, easy to answer, appropriate and could spark engagement.

---

WORKED EXAMPLE

Input:
Subject: Italy
Phrase: recently been to Italy
Relationship: friend
Research budget: you can and should call the `research` tool up to 3 times.
Brainstorm (fixed input):
Subject facet angles:

* Italian cuisine and local food experiences  (context: Italy is commonly associated with pizza, pasta, chocolate, wine, and regional food culture.)
* Major tourist landmarks he visited  (context: Italy's famous attractions include the Colosseum in Rome, the Grand Canal in Venice, and Florence's Duomo.)
* Whether he prioritized famous attractions or more spontaneous exploration of local neighborhoods
* How he handled communication barriers with Italian people
* Compare common stereotypes about Italian culture with his actual experience  (context: Common stereotypes include talking with their hands, being romantic, and being fashionable; Milan, Italy's fashion capital, drives a lot of that reputation.)
  Person angles:
* What originally motivated him to choose Italy as a travel destination  (context: It could have been a first big trip, a return visit, or a destination he had wanted to visit for a long time.)
* Whether the trip changed his future travel goals or made him consider living abroad someday
  Research gaps:
* What recent events in Italy might have touched a tourist's trip?
* What current, friendly topics around Italy are easy to bring up casually?

Research calls made:
research(
query="Italy recent news",
context="I am meeting a friend who has recently been to Italy. I would
like to know some recent Italy news as suitable conversational topics.",
reason="Recent events in Italy may connect to my friend's travel experience.",
)

research(
query="Italy transport strikes travel impact",
context="I am meeting a friend who has recently been to Italy. The first search surfaced widespread strikes; I want enough detail to ask whether his trip was affected.",
reason="A focused follow-up on the most conversation-ready recent event from the first call."
)

research(
query="Italy recent topics",
context="I am meeting a friend who has recently been to Italy. I want current, friendly topics that might connect to what they noticed during the trip.",
reason="Broad recent news can surface topics that are easy to ask about casually."
)

Output:
{{
  "subject": "Italy",
  "framing": "Italy is known for its famous food, ancient history, beautiful cities, and relaxed lifestyle culture. Recently, Italy has been dealing with transport strikes, rising fuel prices, and excitement around Italian tennis star Jannik Sinner winning in Rome.",
  "themes": [
    {{
      "name": "Food",
      "points": [
        {{
          "angle": "Food in Italy",
          "context": "Italy is commonly associated with pizza, chocolate, and wine.",
          "source_url": null
        }}
      ]
    }},
    {{
      "name": "Attractions",
      "points": [
        {{
          "angle": "What landmark he visited",
          "context": "Italy's famous attractions include the Colosseum in Rome, the Grand Canal in Venice, and Florence's Duomo.",
          "source_url": null
        }},
        {{
          "angle": "Did he also wander local streets?",
          "context": null,
          "source_url": null
        }}
      ]
    }},
    {{
      "name": "Culture",
      "points": [
        {{
          "angle": "How did he communicate with Italian people?",
          "context": null,
          "source_url": null
        }},
        {{
          "angle": "The Italian way of being",
          "context": "Italians have a reputation for being expressive, social, and romantic.",
          "source_url": null
        }},
        {{
          "angle": "What drew him to Italy in the first place",
          "context": "First big trip or a return visit?",
          "source_url": null
        }},
        {{
          "angle": "Whether the trip made him want to travel more, or even live abroad someday",
          "context": null,
          "source_url": null
        }}
      ]
    }},
    {{
      "name": "Other",
      "points": [
        {{
          "angle": "Transport strikes affecting the trip",
          "context": "Italy is in the middle of widespread strikes right now — transport plus a recent general strike. Did it affect the trip?",
          "source_url": "https://www.reuters.com/world/europe/italy-strikes/"
        }},
        {{
          "angle": "Whether Italy felt expensive",
          "context": "Italian inflation rose in May, driven largely by higher energy and transport costs.",
          "source_url": "https://www.reuters.com/world/europe/italy-energy/"
        }}
      ]
    }}
  ],
  "reasoning_notes": "Rewrote the brainstorm intents into shorter final angles instead of preserving the original wording: 'Italian cuisine and local food experiences' became 'Food in Italy,' 'major tourist landmarks he visited' became 'What landmark he visited,' and 'spontaneous exploration of local neighborhoods' became 'wandering local streets.' Kept the person-angle intents by asking what drew him to Italy and whether the trip made him want to travel more. Spent all 3 research calls on the gaps: broad recent news, a focused follow-up on transport strikes, and current friendly topics. Added two researched points about transport strikes and whether Italy felt expensive, and folded Sinner's win into the framing."
}}

"""

draft_agent = Agent(
    name="DraftAgent",
    instructions=DRAFT_PROMPT,
    model=model("openai/gpt-4.1"),
    model_settings=ModelSettings(tool_choice="required"),
    tools=[research],
    output_type=PlannerOutput,
)

print("draft_agent defined.")
print(f"  model:       openai/gpt-4.1")
print(f"  tools:       ['research'], tool_choice='required' (>=1 real call guaranteed)")
print(f"  max_turns:   {DRAFT_MAX_TURNS}")

draft_agent defined.
  model:       openai/gpt-4.1
  tools:       ['research'], tool_choice='required' (>=1 real call guaranteed)
  max_turns:   10


In [ ]:
# Cell 5 — Two-stage pipeline helper + code guards
#
# Guards (see plannerAgentImprovements.md):
#   1. research-call count (tool_choice should make 0 impossible; warn anyway)
#   2. URL validator — strip any source_url not present in real research output
#   3. carry-over check — >=2 first-principles points

def _count_research_calls(run_result) -> int:
    n = 0
    for item in run_result.new_items:
        if getattr(item, "type", None) == "tool_call_item":
            raw = item.raw_item
            name = getattr(raw, "name", None) or getattr(getattr(raw, "function", None), "name", None)
            if name == "research":
                n += 1
    return n


def _extract_research_urls(run_result) -> set[str]:
    """Every URL that actually appeared in research tool outputs."""
    urls = set()
    for item in run_result.new_items:
        if getattr(item, "type", None) == "tool_call_output_item":
            text = str(getattr(item, "output", ""))
            for u in re.findall(r"https?://[^\s)\"']+", text):
                urls.add(u.rstrip(".,"))
    return urls


In [ ]:
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    _demo_input = (
        "Subject: stock market\n"
        "Phrase: is interested in the stock market\n"
        "Relationship: friend\n"
        "Research budget: you can and should call the `research` tool up to 3 times for this subject.\n"
        "Brainstorm (fixed input):\n"
        "Subject facet angles:\n"
        "- What kinds of stocks or sectors they follow\n"
        "- How they got started investing\n"
        "Person angles:\n"
        "- What investing means to them (wealth, challenge, learning)\n"
        "Research gaps:\n"
        "- What recent stock market developments are worth bringing up?\n"
    )
    _r = await Runner.run(draft_agent, _demo_input, max_turns=DRAFT_MAX_TURNS)
    _out = _r.final_output
    _sourced = sum(1 for t in _out.themes for pt in t.points if pt.source_url)
    print(f"draft demo: subject={_out.subject!r} themes={len(_out.themes)} sourced={_sourced}")
    for t in _out.themes:
        print(f"  [{t.name}]")
        for pt in t.points:
            print(f"    - {pt.angle}{'  [src]' if pt.source_url else ''}")
